## Notebook testing gear assistant model locally
This model is the merged model and require to be stored locally (see README.md for more info on getting it)

In [1]:
# Constants

BASE_MODEL = "meta-llama/Llama-3.2-3B-Instruct"
PROJECT_NAME = "gear_training"
HF_USER = "yrichard"
DATA_USER = "yrichard"
DATASET_NAME = f"{DATA_USER}/gear_raw"

MAX_SEQUENCE_LENGTH = 768
RUN_NAME = "2026-04-28_13.15.01"

PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}"
HUB_MODEL_NAME = f"{HF_USER}/{PROJECT_RUN_NAME}"

Import Dataset

In [ ]:

import os
from dotenv import load_dotenv
from huggingface_hub import login
from datasets import load_dataset

load_dotenv()

DATA_USER = "yrichard"
DATASET_NAME = f"{DATA_USER}/gear_raw"

hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

dataset = load_dataset(DATASET_NAME)
train = dataset['train']
val = dataset['validation']
test = dataset['test']

Token has not been saved to git credential helper.
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Cannot authenticate through git-credential as no helper is defined on your machine.
You might have to re-authenticate when pushing to the Hugging Face Hub.
Run the following command in your terminal in case you want to set the 'store' credential helper as default.

git config --global credential.helper store

Read https://git-scm.com/book/en/v2/Git-Tools-Credential-Storage for more details.


Test merged model locally

In [8]:
import re

def _parse_messages(prompt: str) -> list[dict]:
    """Extract system/user turns from a Llama chat-template formatted prompt."""
    pattern = r"<\|start_header_id\|>(\w+)<\|end_header_id\|>(.*?)<\|eot_id\|>"
    return [
        {"role": m.group(1), "content": m.group(2).strip()}
        for m in re.finditer(pattern, prompt, re.DOTALL)
        if m.group(1) in ("system", "user")
    ]


In [ ]:
import requests

OLLAMA_URL = "http://host.docker.internal:11434"
OLLAMA_MODEL = "gear-assistant"


def model_predict(item):
    response = requests.post(
        f"{OLLAMA_URL}/api/chat",
        json={
            "model": OLLAMA_MODEL,
            "messages": _parse_messages(item["prompt"]),
            "stream": False,
            "options": {"num_predict": 512, "temperature": 0},
        },
        timeout=60,
    )
    response.raise_for_status()
    return response.json()["message"]["content"]

In [4]:
gear = test[3]
print(gear["prompt"])

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 28 Apr 2026

You are a mountain climbing equipment assistant. Parse the following gear description and return a JSON array. Each element must have exactly three fields:
	- "name": equipment name (string, in french)
	- "quantity": number needed (integer, 1 if unspecified)
	- "notes": "optional" or "mandatory" (translated in french), plus any relevant detail (string, in french)
The name of these equipments are related with the mountain activities. You should only point out personal equipment, for instance quickdraws or rope.
You should include only equipment you're absolutely sure about. Output ONLY the JSON array, no explanation.<|eot_id|><|start_header_id|>user<|end_header_id|>

Gear description:
 * 13 dégaines plus relais et deux friends moyens au cas où.

Pour le passage du toit en artif, deux sangles de 120 et une de 60 nécessaires. Pour ce passage en libre prévoir des gant

In [5]:
print("Answer:")
model_predict(gear)

Answer:


'[{"name": "Dégaines", "quantity": 13, "notes": "facultatif"}, {"name": "Friends / coinceurs", "quantity": 2, "notes": "facultatif"}, {"name": "Sangles", "quantity": 5, "notes": "obligatoire"}]'

In [6]:
from evaluate import evaluate
evaluate(model_predict, test)

### Comparing with frontier model calls

Gemini

In [33]:
# Call Gemini API through google-genai library
from dotenv import load_dotenv
from openai import OpenAI
load_dotenv()

from google import genai
from google.genai import types

client = genai.Client()

def model_predict(item):

    google_api_key = os.getenv('GEMINI_API_KEY')
    gemini = OpenAI(api_key=google_api_key, base_url="https://generativelanguage.googleapis.com/v1beta/openai/")
    model_name = os.getenv("GEMINI_MODEL", "gemini-3-flash-preview")

    response = gemini.chat.completions.create(model=model_name, 
                                              messages=_parse_messages(item["prompt"]))
    return response.choices[0].message.content


In [34]:
gear = test[3]
print(gear["prompt"])

print("Answer:")
model_predict(gear)

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 28 Apr 2026

You are a mountain climbing equipment assistant. Parse the following gear description and return a JSON array. Each element must have exactly three fields:
	- "name": equipment name (string, in french)
	- "quantity": number needed (integer, 1 if unspecified)
	- "notes": "optional" or "mandatory" (translated in french), plus any relevant detail (string, in french)
The name of these equipments are related with the mountain activities. You should only point out personal equipment, for instance quickdraws or rope.
You should include only equipment you're absolutely sure about. Output ONLY the JSON array, no explanation.<|eot_id|><|start_header_id|>user<|end_header_id|>

Gear description:
 * 13 dégaines plus relais et deux friends moyens au cas où.

Pour le passage du toit en artif, deux sangles de 120 et une de 60 nécessaires. Pour ce passage en libre prévoir des gant

'```json\n[\n  {\n    "name": "Dégaines",\n    "quantity": 13,\n    "notes": "obligatoire"\n  },\n  {\n    "name": "Friends moyens",\n    "quantity": 2,\n    "notes": "optionnel"\n  },\n  {\n    "name": "Sangles de 120",\n    "quantity": 2,\n    "notes": "obligatoire"\n  },\n  {\n    "name": "Sangle de 60",\n    "quantity": 1,\n    "notes": "obligatoire"\n  },\n  {\n    "name": "Gants de fissure",\n    "quantity": 1,\n    "notes": "optionnel"\n  },\n  {\n    "name": "Sangles à placer avant le toit",\n    "quantity": 3,\n    "notes": "optionnel, pour éviter le tirage"\n  }\n]\n```'

In [35]:
from evaluate import evaluate
evaluate(model_predict, test)

  0%|          | 0/28 [00:00<?, ?it/s]

F1=0.35 F1=0.40 F1=0.00 F1=0.60 F1=0.50 F1=0.32 F1=0.57 F1=0.55 F1=0.00 F1=0.00 F1=0.00 F1=0.33 F1=0.50 F1=0.60 F1=1.00 F1=0.00 

RateLimitError: Error code: 429 - [{'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash-lite\nPlease retry in 50.511744865s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-2.5-flash

OpenAI

In [ ]:
import openai
from dotenv import load_dotenv

load_dotenv()

openai_api_key = os.getenv('OPENAI_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set - please head to the troubleshooting guide in the setup folder")
    
def model_predict(item):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=_parse_messages(item["prompt"])
    )
    result = response.choices[0].message.content
    return result

OpenAI API Key exists and begins sk-proj-


In [9]:
gear = test[3]
print(gear["prompt"])

print("Answer:")
model_predict(gear)

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 28 Apr 2026

You are a mountain climbing equipment assistant. Parse the following gear description and return a JSON array. Each element must have exactly three fields:
	- "name": equipment name (string, in french)
	- "quantity": number needed (integer, 1 if unspecified)
	- "notes": "optional" or "mandatory" (translated in french), plus any relevant detail (string, in french)
The name of these equipments are related with the mountain activities. You should only point out personal equipment, for instance quickdraws or rope.
You should include only equipment you're absolutely sure about. Output ONLY the JSON array, no explanation.<|eot_id|><|start_header_id|>user<|end_header_id|>

Gear description:
 * 13 dégaines plus relais et deux friends moyens au cas où.

Pour le passage du toit en artif, deux sangles de 120 et une de 60 nécessaires. Pour ce passage en libre prévoir des gant

'[\n  {\n    "name": "dégaines",\n    "quantity": 13,\n    "notes": "obligatoire pour l\'escalade"\n  },\n  {\n    "name": "friends moyens",\n    "quantity": 2,\n    "notes": "obligatoire en cas de besoin"\n  },\n  {\n    "name": "sangles de 120 cm",\n    "quantity": 2,\n    "notes": "obligatoire pour le passage du toit en artif"\n  },\n  {\n    "name": "sangle de 60 cm",\n    "quantity": 1,\n    "notes": "obligatoire pour le passage du toit en artif"\n  },\n  {\n    "name": "gants de fissure",\n    "quantity": 1,\n    "notes": "obligatoire pour le passage en libre"\n  },\n  {\n    "name": "sangles",\n    "quantity": 3,\n    "notes": "obligatoire pour éviter le tirage avant le toit"\n  }\n]'

In [10]:
from evaluate import evaluate
evaluate(model_predict, test)